In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import warnings
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

warnings.filterwarnings("ignore")

In [0]:
SENDER_EMAIL = dbutils.secrets.get(scope='email-scope', key='sender-email')  # change scope, key as per what was set
SENDER_PASSWORD = dbutils.secrets.get(scope='email-scope', key='sender-password').replace(' ', '')  # change scope, key as per what was set
RECEIVER_EMAIL = dbutils.secrets.get(scope='email-scope', key='receiver-email')  # change scope, key as per what was set

In [0]:
def compute_drift_metrics():
    w=Window().orderBy(F.col('obs_date')).rowsBetween(-11, 0)
    pred_df=spark.table('us_macroeconomics_tracker.gold.recession_predictions')
    final_df=pred_df.withColumn('rolling_avg_12m',F.avg(F.col('recession_probability')).over(w)).withColumn('std_dev_12m',F.stddev(F.col('recession_probability')).over(w))
    return final_df

In [0]:
def flag_drift():
    final_df=compute_drift_metrics()
    final_df=final_df.withColumn('drift_flag',F.when(F.abs(F.col('recession_probability') - F.col('rolling_avg_12m')) > 2 * F.col('std_dev_12m'),True).otherwise(False))
    return final_df

In [0]:
def save_monitoring_log():
    final_df=flag_drift()
    final_df=final_df.select('obs_date','recession_probability','rolling_avg_12m','std_dev_12m','drift_flag',F.current_timestamp().alias('checked_at'))
    final_df.write.format("delta").mode("overwrite").saveAsTable('us_macroeconomics_tracker.gold.monitoring_log')
save_monitoring_log()

In [0]:
def send_drift_alert():
    flag_df=spark.table('us_macroeconomics_tracker.gold.monitoring_log')
    flag_df=flag_df.orderBy(F.col('obs_date').desc()).limit(1)
    flag_df=flag_df.where(F.col('drift_flag')==True)
    rows = flag_df.collect()
    print(f"Drift rows found: {len(rows)}")
    if len(rows)>0:
        row = rows[0]
        obs_date = row['obs_date']
        recession_probability = row['recession_probability']
        rolling_avg = row['rolling_avg_12m']
        checked_at=row['checked_at']
        message = MIMEMultipart("alternative")
        message["Subject"] = f"Data Drift Alert — US Macroeconomics Pipeline ran on {checked_at}"
        message["From"] = SENDER_EMAIL
        message["To"] = RECEIVER_EMAIL
        body_lines = []
        for row in rows:
            body_lines.append(f"obs_date:{row['obs_date']} - recession_probability:{row['recession_probability']} - rolling_avg_12m:{row['rolling_avg_12m']} - checked_at:{row['checked_at']}")
        body_text = '\n'.join(body_lines)
        text = f"""\
        Data Drift ALERT!! PIPELINE FAILED on {checked_at}.

        {body_text}

        TO LEARN MORE, NAVIGATE TO YOUR NOTEBOOK AND QUERY:
        SELECT * FROM us_macroeconomics_tracker.gold.monitoring_log WHERE checked_at='{checked_at}' and drift_flag=TRUE
        ORDER BY checked_at desc;
        """
        html = f"""\
            <html>
                <body>
                    <p>Data Drift ALERT!! PIPELINE FAILED on {checked_at}.<br>
                       {body_text}</p>
                    <p> TO LEARN MORE, NAVIGATE TO YOUR NOTEBOOK AND QUERY:<br>
                        SELECT * FROM us_macroeconomics_tracker.gold.monitoring_log WHERE checked_at='{checked_at}' and drift_flag=TRUE
                        ORDER BY checked_at desc;</p>
                </body>
            </html>
        """
        part1 = MIMEText(text, "plain")
        part2 = MIMEText(html, "html")
        message.attach(part1)
        message.attach(part2)
        try:
            with smtplib.SMTP_SSL('smtp.gmail.com', 465) as SMTP_SERVER:
                SMTP_SERVER.login(SENDER_EMAIL, SENDER_PASSWORD)
                SMTP_SERVER.sendmail(SENDER_EMAIL, RECEIVER_EMAIL, message.as_string())
                print('EMAIL SENT!')
        except Exception as e:
            print(e)

   

In [0]:
send_drift_alert()

In [0]:
def check_retrain_trigger():
    drift_df=spark.table('us_macroeconomics_tracker.gold.monitoring_log').orderBy(F.col('obs_date').desc()).limit(3)
    rows=drift_df.collect()
    if len(rows) == 3 and all(row['drift_flag'] == True for row in rows):
        print("Retraining triggered")
        dbutils.notebook.run("/Workspace/Users/vishweshpv767@gmail.com/US-Macroeconomics-Indicators-Pipeline/notebooks/Recession_Predictor", timeout_seconds=3600)
    else:
        print('No Drift detected...')

In [0]:
check_retrain_trigger()

In [0]:
print('Saving Monitoring Log...')